In simple environments, we can use a Q-table to store the value of each state-action pair. However, in a complex environment like the stock market, the number of possible states is virtually infinite, making a Q-table impossible to create or store. DQN solves this by using a neural network as a powerful function approximator. Instead of looking up a value in a table, the network takes the state as input and predicts the Q-values for all possible actions.
DQN combines the principle of reinforcement learning with Power deep neural network.
The core mechanism of DQN is to compute a Q-value for every single possible action (through neural network)and then select the action with the highest value.<br>
DQN is designed for problems with a discrete action not for the continous action space.(Buy/Sell/Hold)


# Advantage: 1)Experience Replay 2)Target Network
1. A key innovation that makes DQN stable is Experience Replay. The agent is equipped with a memory buffer that stores its past experiences, where each experience is a tuple of (state, action, reward, next_state).
So instead of training on experiences as they occur, the agent performs training update form its memory and why this is helpful is beacuse we have a data effciency, it means that we can used same data to perfom training multiple times and agent will extract more value by learning from environment each time.It does also break the sequential corelation in data.

Examples: Lets take the example of Landing Lunar so  assume we are at time t , we push the throttle and it fails to go up,
At t+1 we again do the same push the throttle and again it fails to go up and lets say  we do this more 50 steps and we  we would train the Landing Lunar with this data with every time we push teh throttle so what it will learns from that we shoudl never push the throttle  as it is resulting in bad results and if had no experience replay it will break after sometimes but when we have experience replay we have so many data for same states so in some states it would be failing to go up , maybe go little up , complete up and many more so it will not be biased to think that lunar will not go up.

2.  The training process requires calculating a target value that our main network tries to predict.<br> A problem arises if we use the main network itself to calculate this target,<br> as it creates a "moving target" that changes with every update, leading to oscillations and instability.<br>
    We have two network that is main network and target network:<br>
    Main Network:-> This network is trained at every step and is used to select the agent's actions<br>
    Target Network:-> This is a clone of the main network. Its weights are kept frozen for many steps and it is used only to calculate the target values for training.(like for every 2000 steps we froze the training network and update that weights in target network and then unfroze again. )

# Limitations:<br>
1. DQN main ideas was to compute Q-value for every possible action and select the action with max Q-value dur to this it is only feasible for the  dicreate states and it sucks for continous states for that reason.(Let's say we want to tell when to buy/ sell /hold then DQN is good but when it come like how much buy/sell/hold it is continous and DQN sucks so instad of that we use the DDPG.)
2. So as we know that we select the actions which have the maximum Q value so this can cause the overfitting and maybe<br>
    select some of the action  which essentially is not the best action which is available at that time.
3. I will attach some codes for lunar lander in which we can say that it is hyper sensitivity for the hyperparameters <br>
    we really need to run it a lot of times to get a good agent. so not everytime <br>we will haev time to do that so we have to select the hyperparamter very selcetively.
    


# Inverted pendulum problem


The inverted pendulum example in the Keras documentation was solved using DDPG<br> which is particularly well-suited for problems with continuous action spaces. But if we have to implement this<br> by DQN Then we have to tweak the action  in this<br> 
as DQN is built like that it will compute Q-values for all discrete<br> actions and choose the action which will have best Q-values.<br>
Therefore, in order to apply DQN to this continuous control problem, <br>we must convert the continuous actions into a set of discrete actions.


In [54]:
import gymnasium as gym
import tensorflow as tf
from tensorflow import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

ENV_NAME = "Pendulum-v1"

env = gym.make(ENV_NAME)

NUM_STATES = env.observation_space.shape[0]
UPPER_BOUND = env.action_space.high[0]
LOWER_BOUND = env.action_space.low[0]


GAMMA = 0.99          # Discount factor for future rewards  or how much we care about long term.
LEARNING_RATE = 0.001 

# Replay Buffer parameters
BUFFER_CAPACITY = 50000
BATCH_SIZE = 64

# exploration startegy 
# start with 100% exploration and then decrease it slowly as we go upfront.

EPSILON_START = 1.0         # Start with 100% random actions
EPSILON_MIN = 0.1           # Minimum exploration rate
EPSILON_DECAY = 0.995       # Decay rate for epsilon after each episode


TOTAL_EPISODES = 300
UPDATE_TARGET_EVERY_EPISODES = 10 #How often to update the target network


#Action state Dicretization 
# we don't want very less dicreate states nor more as both are bad so just for saking of taking
# i am taking 11  as num of dicrete actions as actions lies b/w [-2,2]

NUM_DISCRETE_ACTIONS = 11



Below class is agent memory or you can say that it stored the information which is used by us on later onwards for learning purpose<br>


In [55]:
class Buffer:
    def __init__(self,buffer_capacity=50000,batch_size=64,num_states=3,num_actions=1):
        self.buffer_capacity = buffer_capacity
        self.batch_size = batch_size
        self.buffer_counter = 0
        self.buffer_size = 0

        #obs_tuple (state,action , reward , next_state,dones) 
        # converting to 2D arrays for faster computation
        self.state_buffer = np.zeros((self.buffer_capacity,num_states),dtype=np.float32)
        self.action_buffer = np.zeros((self.buffer_capacity,1),dtype=np.int32)
        self.reward_buffer = np.zeros((self.buffer_capacity,1),dtype=np.float32)
        self.next_state_buffer = np.zeros((self.buffer_capacity, num_states),dtype=np.float32)
        self.dones = np.zeros((self.buffer_capacity,1),dtype=np.uint8)


    def record(self,obs_tuple):
        """
            store the experiences in  tuple
        """
        index = self.buffer_counter % self.buffer_capacity
        self.state_buffer[index] = obs_tuple[0]
        self.action_buffer[index] = obs_tuple[1]
        self.reward_buffer[index] = obs_tuple[2]
        self.next_state_buffer[index] = obs_tuple[3]
        self.dones[index] = obs_tuple[4]
        self.buffer_counter += 1

        self.buffer_size = min(self.buffer_size + 1, self.buffer_capacity)

    #Randomly select a batch of previously stored experience tuples 
    #and return them as TensorFlow tensors for training the model
    def sample(self):
        """
        Choose random indices from the valid range of the buffer
        """
        batch_indices = np.random.choice(self.buffer_size, self.batch_size, replace=False)
        states = self.state_buffer[batch_indices]

        # Use .flatten() to remove the unnecessary second dimension (e.g., shape (64,1) -> (64,))
        actions = self.action_buffer[batch_indices].flatten()
        rewards = self.reward_buffer[batch_indices].flatten()
        next_states = self.next_state_buffer[batch_indices]
        dones = self.dones[batch_indices].flatten()

        return states,actions,rewards,next_states,dones

    def __len__(self):
        """Returns the current number of items in the buffer."""
        return self.buffer_size

why we have used here dones here is what actually we need at last is target value from bellman eqution and that 
bellman equation gives target value <br>
target_q_values = rewards + (gamma * max_future_q * (1 - dones)) <br>
if the state is the terminal state then there is no future rewards sp dones = 1<br>
Before the terminal state all the statte will have target = rewwards from that + future rewards
and if in terminal state if we didn't used that dones thing then<br> it would cause some problem  as Q-network will ask<br>, "What is the value of this state?". The network, unaware <br>the game has ended, will predict some value<br> based on its training , which isn't true.


In [56]:
class DQNAgent:
    def __init__(self,num_states , num_actions):
        self.num_states = num_states
        self.num_actions = num_actions

        self.epsilon = EPSILON_START # The start of the Exploration and exploitattion policy

        self.model = self._create_q_model() # The main model
        self.target_model = self._create_q_model() # The target model

         # Ensure target network starts with the same weights as the main network
        self.target_model.set_weights(self.model.get_weights())

                # --- Create the Optimizer and Buffer ---
        self.optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
        # Here we use the Buffer class you just finalized!
        self.buffer = Buffer(BUFFER_CAPACITY, BATCH_SIZE, num_states, num_actions)

    def _create_q_model(self):
        """
        Defines the neural network architecture.
        Input: a state vector.
        Output: a Q-value for each possible discrete action.
        """
        inputs = layers.Input(shape=(self.num_states,))
        layer1 = layers.Dense(64, activation='relu')(inputs)
        layer2 = layers.Dense(64 , activation='relu')(layer1)
        action_values = layers.Dense(self.num_actions , activation='linear')(layer2)

        return keras.Model(inputs=inputs,outputs= action_values) # easy to make models from just input and output just
            #like keras.sequnetial is sied for basic network it is used for keras.model is used for mu;ripe input or multiput output networks

    def policy(self, state):
        """
        Implements the epsilon-greedy policy for choosing an action.
        """
        # With probability epsilon, we explore by choosing a random action
        if np.random.rand() <= self.epsilon:
            return np.random.choice(self.num_actions)
        
        # Otherwise, we exploit by asking our main model for the best action
        q_values = self.model.predict(state, verbose=0)
        return np.argmax(q_values[0]) # this givves inner 1D array 

    def update_target_network(self):
        """
        Copies the weights from the main model to the target model.
        """
        self.target_model.set_weights(self.model.get_weights())

    def learn(self):
        """
        This is the core training logic. It implements the Bellman equation update.
        """
        # 1. Don't learn until the buffer has enough experiences to form a batch
        if len(self.buffer) < BATCH_SIZE:
            return

        #  Sample a random mini-batch of experiences from the buffer
        # This returns pure NumPy arrays, as we designed.
        states, actions, rewards, next_states, dones = self.buffer.sample()
        
        # Calculate the target Q-values using the Bellman equation
        #    target = reward + gamma * max_Q(next_state)
        #    We use the STABLE `target_model` for the future Q-value part.
        future_q_values = self.target_model.predict(next_states, verbose=0)
        max_future_q = np.max(future_q_values, axis=1)
        
        # If 'done' is True (1), the future part is zeroed out.
        target_q_values = rewards + (GAMMA * max_future_q * (1 - dones))

        # 4. Update the main network using gradient descent
        # We create a mask to tell TensorFlow which action's Q-value to update
        masks = tf.one_hot(actions, self.num_actions)

        with tf.GradientTape() as tape: #by using this tensorflow will record the derivation which we nedd in gradient descent
            # Get the Q-values our MAIN model predicts for the states
            all_q_values = self.model(states)
            # Select the Q-value for the action that was actually taken
            q_action = tf.reduce_sum(tf.multiply(all_q_values, masks), axis=1)
            # Calculate the loss between our prediction and the "true" target value
            mse = keras.losses.MeanSquaredError()
            loss = mse(target_q_values, q_action)

        # Calculate the gradients and apply them to the main model's weights
        grads = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        
    # In the DQNAgent class, add this new method
    def decay_epsilon(self):
        if self.epsilon > EPSILON_MIN:
            self.epsilon *= EPSILON_DECAY

    

In [ ]:
if __name__ == "__main__":
    
    env = gym.make(ENV_NAME)
    
    # Create the mapping from our discrete action indices to continuous torque values
    action_map = np.linspace(LOWER_BOUND, UPPER_BOUND,NUM_DISCRETE_ACTIONS)
    
    # Create our DQN Agent
    agent = DQNAgent(NUM_STATES, NUM_DISCRETE_ACTIONS)

    # Lists for tracking rewards to see if our agent is improving
    ep_reward_list = []
   
    
    for ep in range(TOTAL_EPISODES):
        # Reset the environment and get the initial state for the new episode
        prev_state, _ = env.reset()
        # Reshape state to (1, num_states) to be a valid input for the network
        prev_state = np.reshape(prev_state, [1, NUM_STATES])
        
        # Reset the episodic reward counter
        episodic_reward = 0
        while True:
            
            # A. Agent chooses a discrete action based on its epsilon-greedy policy
            discrete_action_index = agent.policy(prev_state)
            
            # B. We translate that discrete index into a continuous torque value
            continuous_action = [action_map[discrete_action_index]]
            
            # C. Take the action in the environment
            state, reward, terminated, truncated, _ = env.step(continuous_action)
            done = terminated or truncated
            state = np.reshape(state, [1, NUM_STATES])
            
            # D. Store this complete experience in the agent's replay buffer
            #    We store the discrete action index, not the continuous value.
            agent.buffer.record((prev_state[0], discrete_action_index, reward, state[0], done))
            
            # E. Agent learns from a random batch of past experiences
            agent.learn()
            
            # Add the single-step reward to our episodic total
            episodic_reward += reward
            
            # Move to the next state
            prev_state = state
            
            # If the episode is over, break the inner loop
            if done:
                break
    
        # Periodically update the target network's weights to match the main network
        if (ep + 1) % UPDATE_TARGET_EVERY_EPISODES == 0:
            agent.update_target_network()
            
        agent.decay_epsilon() # decay the epsilon 
        
        
        ep_reward_list.append(episodic_reward)
        print(ep_reward_list[-1])
        
    



For this assignment, I solved the inverted pendulum problem using a Deep Q-Network, or DQN. Fundamentally, DQN is designed for environments with discrete action spaces, like choosing to buy, sell, or hold. However, the pendulum environment has a continuous action space, requiring a torque between -2.0 and +2.0. To solve this, I adapted the problem by **discretizing the action space**. I converted the continuous range into 11 distinct actions, effectively creating 'preset buttons' for forces like 'full left' (-2.0), 'neutral' (0.0), and 'full right' (+2.0).

My implementation consists of two main components. First, a **Replay Buffer**, which stores the agent's past experiences. During learning, the agent samples a random batch of these experiences, which is a crucial technique called Experience Replay that de-correlates the data and stabilizes training.

Second, the **DQN Agent** itself contains the core logic. I built a neural network that takes the pendulum's state as input and outputs the predicted **Q-value** for each of the 11 discrete actions. To solve the instability of a single network learning on a moving target, I used two networks: a **main network** that learns continuously and makes decisions, and a separate **target network** that is updated periodically to provide a stable learning goal.

The learning process works as follows: at each step, the agent chooses an action using an epsilon-greedy policy for exploration. It then interacts with the environment and stores the experience. To train, it samples a batch from the buffer and calculates a target Q-value using the **Bellman equation** and the stable **target network**. It then computes the Mean Squared Error between this target and the prediction from its **main network**. This error is used to update the main network's weights via gradient descent. Critically, my implementation correctly handles terminal states. By storing a `done` flag with each experience, the Bellman equation correctly sets the target value to just the final reward for any game-ending move, ensuring the agent learns to associate failure with low value.